# **TRIP-Sense:** A Dynamic Travel Planner Using Natural Language Processing & Hybrid Machine Learning

## Team Responsibilities

- **M. Manaswi Priya:**  
  Natural Language Processing & User Intent Module - user input processing, intent extraction, sentiment analysis and over-all team-work functioning

- **G. Praneetha:**  
  Hybrid Machine Learning Recommendation System - content-based filtering, collaborative filtering and hybrid model

- **N. Harikrishna:**  
  Context Awareness & Real-Time Integration - API integration, route optimization and dynamic adaptation

- **B. Pavan Kumar:**  
  Dashboard & Visualization - Streamlit interface, maps, charts and final integration

### Personalized • Context-Aware • Optimized

An AI-powered travel assistant that dynamically generates itineraries
based on user preferences, real-time conditions, and intelligent routing.

## Step 1: NLP & User Intent Module

This module processes natural language user input to extract structured travel preferences such as budget, duration, interests, and destination type. The output of this module is used by the recommendation and itinerary generation systems.

### Step 1.1: Import Required NLP Libraries

This step imports all libraries needed for text preprocessing, intent extraction, and sentiment analysis.

In [81]:
from google.colab import drive
drive.mount('/content/drive')

import re
import nltk
import pandas as pd
import spacy

nltk.download('stopwords')
nltk.download('punkt')

nlp = spacy.load("en_core_web_sm")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


### Step 1.2: Accept User Input

This step takes the travel requirement from the user in natural language format.

In [82]:
user_input = "Plan a 3-day trip to Goa under 20000 with beaches and food"
print(user_input)

Plan a 3-day trip to Goa under 20000 with beaches and food


### Step 1.3: Text Preprocessing

This step cleans the input text by converting it to lowercase, removing punctuation, and preparing it for analysis.

In [83]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

clean_text = preprocess_text(user_input)
print(clean_text)

plan a 3day trip to goa under 20000 with beaches and food


### Step 1.4: Extract Budget and Duration

This step identifies numerical information such as trip duration (days) and budget from the user input.

In [84]:
def extract_budget_duration(text):
    duration = re.search(r'(\d+)\s*day', text)
    budget = re.search(r'(\d+)\ with', text)

    return {
        "duration_days": int(duration.group(1)) if duration else None,
        "budget": int(budget.group(1)) if budget else None
    }

basic_info = extract_budget_duration(clean_text)
basic_info

{'duration_days': 3, 'budget': 20000}

### Step 1.5: Extract User Interests

This step identifies interest keywords such as beaches, hills, food, adventure, etc.

In [85]:
interest_keywords = ["beach", "hill", "mountain", "food", "heritage", "adventure"]

def extract_interests(text):
    return [word for word in interest_keywords if word in text]

interests = extract_interests(clean_text)
interests

['beach', 'food']

### Step 1.6: Extract Destination Name

This step attempts to detect a destination name using Named Entity Recognition.

In [86]:
doc = nlp(user_input)

destination = None
for ent in doc.ents:
    if ent.label_ == "GPE":
        destination = ent.text

destination

### Step 1.7: Sentiment Analysis on Reviews

This step evaluates sentiment of travel reviews to help prioritize positively reviewed destinations.

In [87]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

sample_review = "Amazing beach with great food and atmosphere"
sentiment_score = sia.polarity_scores(sample_review)

sentiment_score

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


{'neg': 0.0, 'neu': 0.388, 'pos': 0.612, 'compound': 0.836}

### Step 1.8: Build Structured User Profile

This step combines all extracted information into a structured format for downstream modules.

In [88]:
user_profile = {
    "destination": destination,
    "duration_days": basic_info["duration_days"],
    "budget": basic_info["budget"],
    "interests": interests
}

user_profile

{'destination': None,
 'duration_days': 3,
 'budget': 20000,
 'interests': ['beach', 'food']}

### Step 1.9: Validate User Profile Output

This step verifies that all required fields are extracted correctly from user input.

In [89]:
assert user_profile["budget"] is not None, "Budget not extracted"
assert user_profile["duration_days"] is not None, "Duration not extracted"
assert isinstance(user_profile["interests"], list), "Interests not in list format"

print("User profile validated successfully")

User profile validated successfully


### Step 1.10: Save User Profile for Downstream Modules

This step saves the structured user profile so it can be used by recommendation, context, and UI modules.

In [90]:
import json

with open("user_profile.json", "w") as f:
    json.dump(user_profile, f)

print("User profile saved successfully")

User profile saved successfully


### Step 1.11: Step 1 Output Interface

The NLP & User Intent module outputs a JSON object with the following structure:

{
  "destination": <string or null>,
  "duration_days": <integer>,
  "budget": <integer>,
  "interests": <list of strings>
}

This output will be consumed by:
- Step 2: Hybrid ML Recommendation System
- Step 3: Context Awareness & Real-Time Integration
- Step 4: Dashboard & Visualization

In [91]:
# from google.colab import files
# files.download("user_profile.json") # download this file if needed!

## Step 2: Hybrid Machine Learning Recommendation System

This module builds a hybrid recommendation system using content-based and collaborative filtering techniques.
It takes the structured user profile from Step 1 and generates ranked travel destination recommendations.

### Step 2.1: Import Required Libraries

This step imports the Python libraries required to build and evaluate the hybrid machine learning recommendation system.

In [ ]:
!pip install "numpy<2"
!pip install scikit-surprise

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
print("Step 2: Hybrid Machine Learning Recommendation System Initialized")

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy

Step 2: Hybrid Machine Learning Recommendation System Initialized


### Step 2.2: Load User Profile and Cleaned Datasets

This step loads the structured user profile generated in Step 1 along with the cleaned datasets required for building the recommendation models.

In [92]:
import json

with open("/content/drive/MyDrive/TRIP-Sense/data/user_profile.json", "r") as f:
    user_profile = json.load(f)

places = pd.read_csv("/content/drive/MyDrive/TRIP-Sense/data/clean_places_with_coords.csv")
reviews = pd.read_csv("/content/drive/MyDrive/TRIP-Sense/data/Reviews.csv")
ratings = pd.read_csv("/content/drive/MyDrive/TRIP-Sense/data/Ratings.csv")

print("User Profile:")
print(user_profile)

print("\nPlaces Dataset:")
display(places.head())

print("\nRatings Dataset:")
display(ratings.head())

User Profile:
{'destination': None, 'duration_days': 3, 'budget': 20000, 'interests': ['beach', 'food']}

Places Dataset:


,place_id,Name,Type,lat,lon,avg_cost,avg_rating
0,1,Baga Beach,beach,15.5553,73.7517,3000,4.5
1,2,Calangute Beach,beach,15.5439,73.7553,2500,4.4
2,3,Anjuna Beach,beach,15.5740,73.7407,2000,4.3
3,4,Fort Aguada,heritage,15.4989,73.7737,500,4.6
4,5,Dudhsagar Falls,nature,15.3144,74.3143,1500,4.7



Ratings Dataset:


,user_id,preferred_categories,budget_range,trip_duration_days,past_places,rating_given
0,U0001,"hill_station,beach,food",medium,3,Jaipur,5
1,U0002,urban,low,6,"Goa,Darjeeling",3
2,U0003,wildlife,high,6,Mysore,3
3,U0004,"urban,adventure,hill_station",medium,6,"Goa,Jaipur",5
4,U0005,"food,religious",low,3,"Manali,Darjeeling",4


### Data Preprocessing (Performed Before Recommendation)

This step cleans and standardizes datasets to ensure reliable model training.

In [94]:
places.dropna(inplace=True)
reviews.dropna(inplace=True)
ratings.dropna(inplace=True)

places["Type"] = places["Type"].str.lower()

places["avg_cost"] = places["avg_cost"].astype(float)
places["avg_rating"] = places["avg_rating"].astype(float)
ratings["rating_given"] = ratings["rating_given"].astype(float)

### Step 2.3: Content-Based Feature Engineering

This step converts destination attributes into numerical feature vectors.
These vectors are used to compute similarity between user preferences and tourist places.

In [95]:
# Clean Type column
places["Type_clean"] = places["Type"].str.lower()

# Create feature matrix from Type
place_features = pd.get_dummies(places["Type_clean"])

place_features.head()

,adventure,beach,city,heritage,hill,nature,temple
0,False,True,False,False,False,False,False
1,False,True,False,False,False,False,False
2,False,True,False,False,False,False,False
3,False,False,False,True,False,False,False
4,False,False,False,False,False,True,False


### Step 2.4: Build User Preference Vector

This step converts the user’s interests into a numerical preference vector.
The vector is aligned with the destination feature matrix and is used for similarity computation.

In [96]:
user_vector = np.zeros(place_features.shape[1])

for interest in user_profile["interests"]:
    interest = interest.lower()
    if interest in place_features.columns:
        user_vector[place_features.columns.get_loc(interest)] = 1

# 🔥 IMPORTANT fallback (fixes your zero problem)
if user_vector.sum() == 0:
    user_vector[:] = 1

user_vector

array([0., 1., 0., 0., 0., 0., 0.])

### Step 2.5: Content-Based Similarity Computation

This step computes similarity scores between the user preference vector and destination feature vectors.
Higher similarity scores indicate better alignment with user interests.

In [97]:
similarity_scores = cosine_similarity(
    user_vector.reshape(1, -1),
    place_features.values
)[0]

places["content_score"] = similarity_scores

### Step 2.6: Prepare Data for Collaborative Filtering

This step prepares the user-item rating data for collaborative filtering models.
The dataset is converted into the required format for training SVD and KNN models.

In [98]:
reader = Reader(rating_scale=(1, 5))

data = Dataset.load_from_df(ratings[["user_id", "preferred_categories", "rating_given"]],reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("Collaborative filtering data prepared successfully")

Collaborative filtering data prepared successfully


### Step 2.7: Train SVD Collaborative Filtering Model

This step trains a Singular Value Decomposition (SVD) model using the user-item rating data.
The model learns latent factors to predict user preferences for unseen destinations.

In [99]:
svd_model = SVD(random_state=42)

svd_model.fit(trainset)

svd_predictions = svd_model.test(testset)
rmse = accuracy.rmse(svd_predictions)

print("SVD model trained successfully")

RMSE: 0.8281
SVD model trained successfully


### Step 2.8: Train KNN Collaborative Filtering Model

This step trains a K-Nearest Neighbors (KNN) model to identify similar users based on rating patterns.
The model recommends destinations by leveraging preferences of similar users.

In [100]:
sim_options = {
    "name": "cosine",
    "user_based": True
}

knn_model = KNNBasic(sim_options=sim_options)

knn_model.fit(trainset)

knn_predictions = knn_model.test(testset)
rmse_knn = accuracy.rmse(knn_predictions)

print("KNN model trained successfully")

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 0.8258
KNN model trained successfully


### Step 2.9: Generate Collaborative Filtering Scores

This step predicts user ratings for each destination using the trained collaborative filtering model.
These predicted scores represent user preference learned from similar users.

In [102]:
import random

places["cf_score"] = places["place_id"].apply(
    lambda x: svd_model.predict(
        random.choice(ratings["user_id"].unique()),
        x
    ).est
)

places[["Name", "cf_score"]].head()

,Name,cf_score
0,Baga Beach,3.980000
1,Calangute Beach,3.980000
2,Anjuna Beach,3.986946
3,Fort Aguada,3.970542
4,Dudhsagar Falls,3.973354


### Step 2.10: Hybrid Recommendation Score

This step combines content-based and collaborative filtering scores
to generate a final hybrid recommendation score for each destination.

In [103]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

places["content_score_norm"] = scaler.fit_transform(
    places[["content_score"]]
)

places["cf_score_norm"] = scaler.fit_transform(
    places[["cf_score"]]
)

places["hybrid_score"] = (
    0.6 * places["content_score_norm"] +
    0.4 * places["cf_score_norm"]
)

places[["Name", "content_score", "cf_score", "hybrid_score"]].head()

,Name,content_score,cf_score,hybrid_score
0,Baga Beach,1.0,3.980000,0.817412
1,Calangute Beach,1.0,3.980000,0.817412
2,Anjuna Beach,1.0,3.986946,0.831439
3,Fort Aguada,0.0,3.970542,0.198312
4,Dudhsagar Falls,0.0,3.973354,0.203990


### Step 2.11: Apply User Constraints

This step filters hybrid recommendations based on user constraints such as budget and trip duration.
Only feasible destinations are retained for final recommendation.

In [106]:
filtered_places = places[
    places["avg_cost"] <= user_profile["budget"]]

filtered_places = filtered_places.sort_values(
    "hybrid_score",
    ascending=False)

filtered_places[["Name", "avg_cost", "hybrid_score"]].head()

,Name,avg_cost,hybrid_score
2,Anjuna Beach,2000.0,0.831439
0,Baga Beach,3000.0,0.817412
1,Calangute Beach,2500.0,0.817412
25,Alleppey Beach,1500.0,0.789238
8,Marina Beach,1000.0,0.642320


### Step 2.12: Save Recommendation Output

This step saves the final ranked list of recommended destinations.
The output of this module is used by the context-aware system and the visualization dashboard.

In [107]:
final_recommendations = filtered_places[["Name", "Type", "avg_cost", "hybrid_score"]]

final_recommendations.to_csv("recommended_places.csv", index=False)

print("Recommended places saved successfully")
final_recommendations.head()

Recommended places saved successfully


,Name,Type,avg_cost,hybrid_score
2,Anjuna Beach,beach,2000.0,0.831439
0,Baga Beach,beach,3000.0,0.817412
1,Calangute Beach,beach,2500.0,0.817412
25,Alleppey Beach,beach,1500.0,0.789238
8,Marina Beach,beach,1000.0,0.642320


In [ ]:
# from google.colab import files
# files.download("recommended_places.csv") # download this file if needed!

## Step 3: Context Awareness & Real-Time Integration

This module enhances travel recommendations by incorporating real-time contextual information
such as weather, location, time, and external events.
It adapts the recommended itinerary dynamically based on changing conditions.

In [108]:
print("Step 3: Context Awareness & Real-Time Integration initialized")

Step 3: Context Awareness & Real-Time Integration initialized


### Step 3.1: Import Required Libraries

This step imports the libraries required for real-time context integration,
including weather data handling, location processing, and routing support.

In [109]:
import requests

from datetime import datetime

import pandas as pd
import numpy as np

# Graph and routing support
import heapq

print("Context Awareness libraries imported successfully")

Context Awareness libraries imported successfully


### Step 3.2: Weather API Integration

This step integrates a weather API to fetch real-time weather information
for a given location. The retrieved data will be used later to adapt travel
recommendations based on weather conditions.

In [112]:
WEATHER_API_KEY = "a33e34339d132ff2d76e0e59de3203f5"

def get_weather(city):
    """
    Fetch current weather data for a given city.
    """
    url = (
        f"https://api.openweathermap.org/data/2.5/weather?"
        f"q={city}&appid={WEATHER_API_KEY}&units=metric")
    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        return {
            "temperature": data["main"]["temp"],
            "weather": data["weather"][0]["main"],
            "description": data["weather"][0]["description"]
        }
    else:
        return None

weather_info = get_weather("Goa")
print(weather_info)

{'temperature': 26.21, 'weather': 'Clouds', 'description': 'broken clouds'}


### Step 3.3: Context-Based Decision Logic (Indoor / Outdoor)

This step applies rule-based logic using real-time weather data
to decide whether indoor or outdoor activities should be prioritized.

In [113]:
def decide_activity_type(weather_info):
    """
    Decide preferred activity type based on weather conditions.
    """
    if weather_info is None:
        return "unknown"

    condition = weather_info["weather"].lower()

    if condition in ["rain", "thunderstorm", "drizzle"]:
        return "indoor"
    else:
        return "outdoor"

# example
activity_preference = decide_activity_type(weather_info)
print("Preferred activity type:", activity_preference)

Preferred activity type: outdoor


### Step 3.4: Integrate Context with Recommendations

This step integrates real-time contextual decisions with the hybrid
recommendations by prioritizing destinations that match the preferred
activity type (indoor or outdoor).

In [114]:
ACTIVITY_TYPE_MAP = {
    "beach": "outdoor",
    "hill": "outdoor",
    "mountain": "outdoor",
    "adventure": "outdoor",
    "nature": "outdoor",

    "heritage": "indoor",
    "museum": "indoor",
    "food": "indoor",
    "shopping": "indoor",
    "cafe": "indoor"
}

filtered_places["activity_type"] = filtered_places["Type"].str.lower().map(ACTIVITY_TYPE_MAP)
filtered_places["activity_type"].fillna("outdoor", inplace=True)

filtered_places[["Name", "Type", "activity_type"]].head()

def apply_context_filter(recommendations, activity_type):
    """
    Filter recommendations based on weather-based activity preference.
    """
    if activity_type in ["indoor", "outdoor"]:
        return recommendations[recommendations["activity_type"] == activity_type]
    else:
        return recommendations

context_aware_recommendations = apply_context_filter(filtered_places, activity_preference)

context_aware_recommendations[["Name", "Type", "activity_type", "hybrid_score"]].head()

/tmp/ipykernel_19422/35483145.py:16: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





,Name,Type,activity_type,hybrid_score
2,Anjuna Beach,beach,outdoor,0.831439
0,Baga Beach,beach,outdoor,0.817412
1,Calangute Beach,beach,outdoor,0.817412
25,Alleppey Beach,beach,outdoor,0.789238
8,Marina Beach,beach,outdoor,0.642320


### Step 3.5: Route Optimization - Graph Setup

This step sets up a graph representation of destinations.
The graph will be used later with shortest path algorithms
such as Dijkstra or A* to compute optimal travel routes.

In [115]:
graph = {}

def add_edge(graph, src, dest, weight):
    """
    Add an edge between two destinations with a given distance.
    """
    if src not in graph:
        graph[src] = []
    if dest not in graph:
        graph[dest] = []

    graph[src].append((dest, weight))
    graph[dest].append((src, weight))

places_list = filtered_places["Name"].tolist()

for i in range(len(places_list) - 1):
    src = places_list[i]
    dest = places_list[i + 1]

    distance = np.random.randint(5, 20)

    add_edge(graph, src, dest, distance)


print("Graph created successfully")
graph

Graph created successfully


{'Anjuna Beach': [('Baga Beach', 10)],
 'Baga Beach': [('Anjuna Beach', 10), ('Calangute Beach', 17)],
 'Calangute Beach': [('Baga Beach', 17), ('Alleppey Beach', 6)],
 'Alleppey Beach': [('Calangute Beach', 6), ('Marina Beach', 7)],
 'Marina Beach': [('Alleppey Beach', 7), ('Marine Drive', 13)],
 'Marine Drive': [('Marina Beach', 13), ('Rishikesh River Rafting', 14)],
 'Rishikesh River Rafting': [('Marine Drive', 14),
  ('Haridwar Ganga Aarti', 12)],
 'Haridwar Ganga Aarti': [('Rishikesh River Rafting', 12),
  ('Victoria Memorial', 10)],
 'Victoria Memorial': [('Haridwar Ganga Aarti', 10),
  ('Sundarbans Forest', 14)],
 'Sundarbans Forest': [('Victoria Memorial', 14), ('Coorg Hills', 19)],
 'Coorg Hills': [('Sundarbans Forest', 19), ('Munnar Tea Gardens', 14)],
 'Munnar Tea Gardens': [('Coorg Hills', 14), ('Howrah Bridge', 9)],
 'Howrah Bridge': [('Munnar Tea Gardens', 9), ('Shillong Hills', 13)],
 'Shillong Hills': [('Howrah Bridge', 13), ('Manali Hills', 17)],
 'Manali Hills': [('Sh

### Step 3.6: Optimal Route Computation Using Dijkstra’s Algorithm

This step applies Dijkstra’s shortest path algorithm on the destination graph
to compute the most efficient travel route between two locations.

In [116]:
def dijkstra(graph, start):
    """
    Compute shortest paths from start node to all other nodes.
    """
    distances = {node: float('inf') for node in graph}
    distances[start] = 0

    priority_queue = [(0, start)]

    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        if current_distance > distances[current_node]:
            continue
        for neighbor, weight in graph[current_node]:
            distance = current_distance + weight
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(priority_queue, (distance, neighbor))
    return distances

# example
shortest_routes = dijkstra(graph, places_list[0])
shortest_routes

{'Anjuna Beach': 0,
 'Baga Beach': 10,
 'Calangute Beach': 27,
 'Alleppey Beach': 33,
 'Marina Beach': 40,
 'Marine Drive': 53,
 'Rishikesh River Rafting': 67,
 'Haridwar Ganga Aarti': 79,
 'Victoria Memorial': 89,
 'Sundarbans Forest': 103,
 'Coorg Hills': 122,
 'Munnar Tea Gardens': 136,
 'Howrah Bridge': 145,
 'Shillong Hills': 158,
 'Manali Hills': 175,
 'Elephanta Caves': 189,
 'Ooty Lake': 200,
 'Jaipur City Palace': 212,
 'Dudhsagar Falls': 221,
 'Fort Aguada': 229,
 'Kerala Backwaters': 245,
 'Hawa Mahal': 255,
 'Solang Valley': 261,
 'Mysore Palace': 271,
 'Gateway of India': 289,
 'Meenakshi Temple': 301,
 'Jaisalmer Fort': 311,
 'Varanasi Ghats': 330,
 'Kashi Vishwanath Temple': 340,
 'Rohtang Pass': 356}

### Step 3.7: Context-Aware Route Selection

This step selects an optimal visiting sequence for recommended destinations
by combining context-aware recommendations with shortest path distances.

In [117]:
def select_best_route(recommendations, graph, start_location):
    """
    Select a simple optimal route based on shortest distances from start location.
    """
    distances = dijkstra(graph, start_location)
    recommendations = recommendations.copy()
    recommendations["distance_from_start"] = recommendations["Name"].map(distances)
    return recommendations.sort_values("distance_from_start")

# example
final_route_plan = select_best_route(
    context_aware_recommendations,
    graph,
    start_location=places_list[0])

final_route_plan[["Name", "activity_type", "distance_from_start"]].head()

,Name,activity_type,distance_from_start
2,Anjuna Beach,outdoor,0
0,Baga Beach,outdoor,10
1,Calangute Beach,outdoor,27
25,Alleppey Beach,outdoor,33
8,Marina Beach,outdoor,40


## Step 4: Dashboard & Visualization

This module presents the final travel recommendations, routes, and insights
through an interactive dashboard. It integrates outputs from all previous modules
and provides a user-friendly interface for exploration and decision-making.

In [118]:
print("Step 4: Dashboard & Visualization initialized")

Step 4: Dashboard & Visualization initialized


### Step 4.1: Prepare Data for Visualization

This step prepares the final recommendation and route data for visualization.
It combines relevant fields required for display in the dashboard.

In [119]:
dashboard_data = final_route_plan.copy()

dashboard_data = dashboard_data[
    ["Name", "Type", "activity_type", "hybrid_score",
     "distance_from_start", "lat", "lon"]
]

dashboard_data.reset_index(drop=True, inplace=True)

print("Dashboard data prepared successfully")
dashboard_data.head()

Dashboard data prepared successfully


,Name,Type,activity_type,hybrid_score,distance_from_start,lat,lon
0,Anjuna Beach,beach,outdoor,0.831439,0,15.5740,73.7407
1,Baga Beach,beach,outdoor,0.817412,10,15.5553,73.7517
2,Calangute Beach,beach,outdoor,0.817412,27,15.5439,73.7553
3,Alleppey Beach,beach,outdoor,0.789238,33,9.4981,76.3388
4,Marina Beach,beach,outdoor,0.642320,40,13.0500,80.2824


### Step 4.2: Display Dashboard Table

This step displays the final recommendations and route plan
in a structured table format for user interpretation.

In [120]:
print("TRIP-Sense Personalized Travel Plan")
display(dashboard_data)

TRIP-Sense Personalized Travel Plan


,Name,Type,activity_type,hybrid_score,distance_from_start,lat,lon
0,Anjuna Beach,beach,outdoor,0.831439,0,15.5740,73.7407
1,Baga Beach,beach,outdoor,0.817412,10,15.5553,73.7517
2,Calangute Beach,beach,outdoor,0.817412,27,15.5439,73.7553
3,Alleppey Beach,beach,outdoor,0.789238,33,9.4981,76.3388
4,Marina Beach,beach,outdoor,0.642320,40,13.0500,80.2824
5,Marine Drive,city,outdoor,0.400000,53,18.9430,72.8238
6,Rishikesh River Rafting,adventure,outdoor,0.395244,67,30.0869,78.2676
7,Sundarbans Forest,nature,outdoor,0.355741,103,21.9497,89.1833
8,Coorg Hills,hill,outdoor,0.243024,122,12.3375,75.8069
9,Munnar Tea Gardens,nature,outdoor,0.240073,136,10.0889,77.0595


### Step 4.3: Visualize Recommendations Using Charts

This step visualizes the recommended destinations using bar charts
to provide an intuitive understanding of ranking and scores.

In [121]:
print(dashboard_data.head())
print("\nData types:\n", dashboard_data.dtypes)
print("\nCheck NaNs:\n", dashboard_data["hybrid_score"].isna().sum())

dashboard_data["hybrid_score"] = pd.to_numeric(
    dashboard_data["hybrid_score"],
    errors="coerce"
)

dashboard_data["hybrid_score"].fillna(0, inplace=True)

dashboard_data["hybrid_score"].describe()

              Name   Type activity_type  hybrid_score  distance_from_start  \
0     Anjuna Beach  beach       outdoor      0.831439                    0   
1       Baga Beach  beach       outdoor      0.817412                   10   
2  Calangute Beach  beach       outdoor      0.817412                   27   
3   Alleppey Beach  beach       outdoor      0.789238                   33   
4     Marina Beach  beach       outdoor      0.642320                   40   

       lat      lon  
0  15.5740  73.7407  
1  15.5553  73.7517  
2  15.5439  73.7553  
3   9.4981  76.3388  
4  13.0500  80.2824  

Data types:
 Name                    object
Type                    object
activity_type           object
hybrid_score           float64
distance_from_start      int64
lat                    float64
lon                    float64
dtype: object

Check NaNs:
 0


/tmp/ipykernel_19422/998889962.py:10: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





,hybrid_score
count,20.000000
mean,0.346288
std,0.280947
min,0.000000
25%,0.200302
50%,0.231735
75%,0.460580
max,0.831439


In [122]:
import plotly.express as px

fig = px.bar(
    dashboard_data,
    x="Name",
    y="hybrid_score",
    color="activity_type",
    title="Top Recommended Destinations",
)

fig.show()

### Step 4.4: Interactive Map Visualization

This step visualizes recommended destinations on an interactive map.
It displays locations and routes to enhance user understanding.

In [123]:
import folium

map_center = [
    dashboard_data["lat"].mean(),
    dashboard_data["lon"].mean()
]

travel_map = folium.Map(location=map_center, zoom_start=5)

# Add real markers
for _, row in dashboard_data.iterrows():
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=f"{row['Name']} ({row['activity_type']})",
        tooltip=row["Name"]
    ).add_to(travel_map)

# Draw route
coordinates = dashboard_data[["lat", "lon"]].values.tolist()

folium.PolyLine(
    locations=coordinates,
    color="blue",
    weight=4
).add_to(travel_map)

travel_map